In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder


train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(train.shape)
train.head()


print(train.isnull().sum())

print("\nSurvival rate by sex:")
print(train.groupby('Sex')['Survived'].mean())

print("\nSurvival rate by class:")
print(train.groupby('Pclass')['Survived'].mean())


def preprocess(df):
    df = df.copy()

    # --- Fix missing values (pandas 2.x compatible) ---
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    # --- Extract Title from Name ---
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
         'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
    )
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    df['Title'] = df['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})
    df['Title'] = df['Title'].fillna(0)

    # --- Age groups ---
    df['Age'] = df['Age'].astype(float)
    df['AgeBand'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                           labels=[0, 1, 2, 3, 4])
    df['AgeBand'] = df['AgeBand'].astype(float).astype(int)

    # --- Fare groups ---
    df['Fare'] = df['Fare'].astype(float)
    df['FareBand'] = pd.qcut(df['Fare'], q=4, labels=[0, 1, 2, 3], duplicates='drop')
    df['FareBand'] = df['FareBand'].astype(float).astype(int)

    # --- Family features ---
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['FamilyGroup'] = pd.cut(df['FamilySize'], bins=[0, 1, 4, 20],
                                labels=[0, 1, 2])
    df['FamilyGroup'] = df['FamilyGroup'].astype(float).astype(int)

    # --- Encode categorical columns ---
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

    # --- Select features ---
    features = [
        'Pclass', 'Sex', 'Age', 'Fare', 'Embarked',
        'FamilySize', 'IsAlone', 'FamilyGroup',
        'Title', 'AgeBand', 'FareBand'
    ]
    return df[features]

X_train = preprocess(train)
y_train = train['Survived']
X_test  = preprocess(test)

print("Features:", X_train.columns.tolist())
print("Shape:", X_train.shape)
X_train.head()


# Train a quick model just to see feature importance
quick_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
quick_model.fit(X_train, y_train)

# Plot feature importance
importances = pd.Series(quick_model.feature_importances_, index=X_train.columns)
importances.sort_values().plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Feature importance')
plt.tight_layout()
plt.show()


# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 6, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# GridSearchCV tries every combination and picks the best
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,      # use all CPU cores to speed it up
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")


# Best Random Forest from grid search
best_rf = grid_search.best_estimator_
rf_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='accuracy')

# Gradient Boosting — often beats Random Forest on Titanic
gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)
gb_scores = cross_val_score(gb, X_train, y_train, cv=5, scoring='accuracy')

print(f"Random Forest   CV: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")
print(f"Gradient Boost  CV: {gb_scores.mean():.4f} ± {gb_scores.std():.4f}")

# Pick the better one
if gb_scores.mean() > rf_scores.mean():
    print("\nUsing Gradient Boosting")
    best_model = gb
else:
    print("\nUsing Random Forest")
    best_model = best_rf

best_model.fit(X_train, y_train)


predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)

print(f"Saved! Rows: {len(submission)}")
print(f"Predicted survivors: {predictions.sum()} out of {len(predictions)}")
submission.head(10)

(891, 12)
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Survival rate by sex:
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Survival rate by class:
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


C:\Users\HIMONK\AppData\Local\Temp\ipykernel_8160\600943209.py:30: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\HIMONK\AppData\Local\Temp\ipykernel_8160\600943209.py:31: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignme

ValueError: Cannot convert float NaN to integer